# SGD vs. IDBD vs. IDBD mit β-Momentum

Momentum wirkt **nur auf die Log-Lernraten**: `v = μ*v - grad*h`, `β += meta_lr*v`. Gewichte und h werden danach nach der bisherigen IDBD-Regel aktualisiert. Das ist eine heuristische Meta-Update-Variante, keine exakte Differentiation durch Momentum.

Alle drei Modelle sehen dieselben Batches und starten mit identischen Gewichten. Nur Feature 0 ist relevant; alle übrigen Features sind Distraktoren (unabhängige Bernoulli(0,01)-Variablen). Das Target enthält zusätzlich Gauß-Rauschen mit Varianz 5 und seltene Impulse. Distraktor-Gewichte sollten gegen 0 gehen, das hilfreiche Gewicht gegen 1.

Wir betrachten RMS, einzelne Gewichte, die Verteilung und einzelne Verläufe der Lernraten sowie Fehler auf einem unabhängigen sauberen Teststream. Schnell sinkende Lernraten allein sind kein Erfolg: Sie können falsche Gewichte einfrieren.

Training, Test und Lernraten verwenden die gemeinsame Schnittstelle aus `src.main`. Die Modell- und Optimizer-Auswahl erfolgt hier im Notebook; MLflow ist deaktiviert.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'src' / 'optim').is_dir():
    raise RuntimeError('Notebook vom Repository-Root oder aus notebooks/ starten.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import copy

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import nn

from src.data import DataStream
from src.main import train, evaluate, learning_rates
from src.optim import IDBD, IDBDMomentum

%matplotlib inline

plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True})


In [10]:
SEED = 1234
INPUT_DIM = 128
ITERATIONS = 2**18
BATCH_SIZE = 32
INITIAL_LR = 0.01
META_LR = 0.1
BETA_MOMENTUM = 0.9
RECORD_EVERY = 50  # Vollständige Gewichte/LRs nur alle N Updates speichern.
TEST_ITERATIONS = 100
TEST_BATCH_SIZE = 128
RUN_MATCHED_META_LR = False  # Optionaler zweiter, vollständig frischer Vergleich.

# Momentum verstärkt konstante Meta-Gradienten langfristig um 1/(1-μ).
# Im Hauptvergleich bleibt meta_lr gleich; μ=0.9 bedeutet ca. Faktor 10.
# Der optionale Vergleich reduziert NUR die Momentum-meta_lr um (1-μ).
assert INPUT_DIM >= 2
assert ITERATIONS > 0 and BATCH_SIZE > 0 and RECORD_EVERY > 0
assert TEST_ITERATIONS > 0 and TEST_BATCH_SIZE > 0
assert 0 <= BETA_MOMENTUM < 1


In [ ]:
def compare(*, seed, input_dim, iterations, batch_size, initial_lr, meta_lr,
            beta_momentum, momentum_meta_lr, record_every,
            test_iterations, test_batch_size):
    # Modell-/Optimizer-Auswahl bleibt beim Aufrufer, Training und Test in src.main.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        initial_model = nn.Linear(input_dim, 1, bias=False)
    models = {name: copy.deepcopy(initial_model)
              for name in ('SGD', 'IDBD', 'IDBD + β-momentum')}
    optimizers = {
        'SGD': torch.optim.SGD(models['SGD'].parameters(), lr=initial_lr),
        'IDBD': IDBD(models['IDBD'].parameters(), initial_lr=initial_lr, meta_lr=meta_lr),
        'IDBD + β-momentum': IDBDMomentum(models['IDBD + β-momentum'].parameters(),
            initial_lr=initial_lr, meta_lr=momentum_meta_lr, beta_momentum=beta_momentum),
    }
    trace = {name: {'weights': [], 'rates': [], 'loss_ma_200': []} for name in models}
    steps = []

    @torch.no_grad()
    def record(step, metrics_by_model=None):
        steps.append(step)
        for name, model in models.items():
            trace[name]['weights'].append(model.weight.flatten().cpu().numpy().copy())
            trace[name]['rates'].append(
                learning_rates(model, optimizers[name]).cpu().numpy().copy())
            trace[name]['loss_ma_200'].append(
                metrics_by_model[name]['loss_ma_200'] if metrics_by_model else np.nan)

    def report(step, metrics_by_model):
        if step % record_every == 0 or step == iterations:
            record(step, metrics_by_model)
        if step % max(1, iterations // 10) == 0 or step == iterations:
            print(f'{step}/{iterations} updates')

    record(0)
    # Kein MLflow; nur gesampelte Feature-Traces, keine vollständige History.
    # Auch die DataLoader dürfen den äußeren Torch-RNG-Zustand nicht verändern.
    with torch.random.fork_rng(devices=[]):
        _, _, _, train_metrics, _ = train(
            DataStream(input_dim=input_dim, seed=seed),
            iterations=iterations, batch_size=batch_size,
            models=models, optimizers=optimizers,
            progress_callback=report, record_history=False, log_mlflow=False,
        )
        metrics = evaluate(
            models, DataStream(input_dim=input_dim, seed=seed + 1),
            iterations=test_iterations, batch_size=test_batch_size,
        ).rename(columns={
            'noisy_mse': 'test_noisy_mse',
            'clean_mse': 'test_clean_mse',
            'active_clean_mse': 'test_active_clean_mse',
        })
    for name, values in trace.items():
        for key in values:
            values[key] = np.asarray(values[key])
        weights, rates = values['weights'][-1], values['rates'][-1]
        metrics.loc[name, 'useful_weight'] = weights[0]
        metrics.loc[name, 'useful_lr'] = rates[0]
        metrics.loc[name, 'distractor_weight_rms'] = np.sqrt(np.mean(weights[1:] ** 2))
        metrics.loc[name, 'distractor_lr_mean'] = rates[1:].mean()
        metrics.loc[name, 'distractor_lr_median'] = np.median(rates[1:])
    return {'models': models, 'optimizers': optimizers, 'trace': trace,
            'steps': np.asarray(steps), 'examples': np.asarray(steps) * batch_size,
            'train_metrics': train_metrics, 'metrics': metrics}


In [ ]:
config = dict(seed=SEED, input_dim=INPUT_DIM, iterations=ITERATIONS, batch_size=BATCH_SIZE,
              initial_lr=INITIAL_LR, meta_lr=META_LR, beta_momentum=BETA_MOMENTUM,
              record_every=RECORD_EVERY, test_iterations=TEST_ITERATIONS,
              test_batch_size=TEST_BATCH_SIZE)
results = compare(**config, momentum_meta_lr=META_LR)
display(results['metrics'])


## Distraktor-Übersicht und hilfreiches Feature

Das Lernraten-Band zeigt das 10.–90. Perzentil **über Features**, kein Konfidenzintervall. Die Mittelwerte können von wenigen großen Lernraten dominiert werden; deshalb zusätzlich Median und Quantile. Alle Gewichte und Lernraten sind post-update, inklusive gemeinsamer Initialisierung bei Update 0.

In [ ]:
def plot_overview(result, title):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
    for name, values in result['trace'].items():
        weights, rates = values['weights'], values['rates']
        examples = result['examples']
        axes[0, 0].plot(examples, np.sqrt(np.mean(weights[:, 1:] ** 2, axis=1)), label=name)
        q10, median, q90 = np.quantile(rates[:, 1:], [0.1, 0.5, 0.9], axis=1)
        line, = axes[0, 1].plot(examples, median, label=f'{name}: median')
        axes[0, 1].fill_between(examples, q10, q90, color=line.get_color(), alpha=0.15)
        axes[0, 1].plot(examples, rates[:, 1:].mean(axis=1), '--',
                        color=line.get_color(), label=f'{name}: mean')
        axes[1, 0].plot(examples, weights[:, 0], label=name)
        axes[1, 1].plot(examples, rates[:, 0], label=name)
    axes[0, 0].set(title='Distractor weight RMS (ideal: 0)', ylabel='RMS', yscale='log')
    axes[0, 1].set(title='Distractor learning rates: median, mean, 10–90%',
                   ylabel='Learning rate', yscale='log')
    axes[1, 0].set(title='Useful feature weight', ylabel='Weight')
    axes[1, 0].axhline(1, color='black', linestyle=':', label='Ideal weight')
    axes[1, 1].set(title='Useful feature learning rate', ylabel='Learning rate', yscale='log')
    for ax in axes.flat:
        ax.set_xlabel('Examples processed')
        ax.legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

plot_overview(results, 'Same meta_lr: SGD vs. IDBD vs. β-momentum')


## Einzelne Distraktoren

In allen drei Modellen werden dieselben Feature-Indizes gezeigt (erste sechs Distraktoren, ohne Auswahl nach Ergebnis). Die Gewichte sind vorzeichenbehaftet, die Lernraten logarithmisch. Momentum kann β auch bei einem aktuell inaktiven Feature weiterbewegen, solange dessen Buffer nicht abgeklungen ist.

In [ ]:
selected_features = np.arange(1, min(INPUT_DIM, 7))
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True,
                         sharey='row', constrained_layout=True)
for column, (name, values) in enumerate(results['trace'].items()):
    for feature in selected_features:
        axes[0, column].plot(results['examples'], values['weights'][:, feature],
                             label=f'x[{feature}]', alpha=0.8)
        axes[1, column].plot(results['examples'], values['rates'][:, feature],
                             label=f'x[{feature}]', alpha=0.8)
    axes[0, column].set_title(name)
    axes[0, column].axhline(0, color='black', linestyle=':')
    axes[0, column].set_ylabel('Distractor weight')
    axes[1, column].set(ylabel='Distractor learning rate', yscale='log',
                        xlabel='Examples processed')
    axes[0, column].legend(fontsize=8)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
all_weights = np.concatenate([v['weights'][-1, 1:] for v in results['trace'].values()])
all_log_rates = np.concatenate([np.log10(v['rates'][-1, 1:]) for v in results['trace'].values()])
weight_bins = np.histogram_bin_edges(all_weights, bins=35)
rate_bins = np.histogram_bin_edges(all_log_rates, bins=35)
for name, values in results['trace'].items():
    axes[0].hist(values['weights'][-1, 1:], bins=weight_bins, histtype='step', label=name)
    axes[1].hist(np.log10(values['rates'][-1, 1:]), bins=rate_bins, histtype='step', label=name)
axes[0].axvline(0, color='black', linestyle=':')
axes[0].set(title='Final distractor weights', xlabel='Weight', ylabel='Feature count')
axes[1].set(title='Final distractor learning rates', xlabel='log10(learning rate)', ylabel='Feature count')
for ax in axes:
    ax.legend()
plt.show()

final_features = pd.DataFrame({'feature': np.arange(INPUT_DIM),
                               'role': ['useful'] + ['distractor'] * (INPUT_DIM - 1)})
for name, values in results['trace'].items():
    final_features[f'{name}: weight'] = values['weights'][-1]
    final_features[f'{name}: lr'] = values['rates'][-1]
display(final_features.head(12))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
for name, values in results['trace'].items():
    axes[0].plot(results['examples'][1:], values['loss_ma_200'][1:], label=name)
axes[0].set(title='Pre-update half-squared loss (200-update MA)',
            xlabel='Examples processed', ylabel='Loss')
axes[0].legend()
for ax, metric, title in zip(axes[1:], ['test_clean_mse', 'test_active_clean_mse'],
                            ['Independent clean-target MSE', 'Clean MSE when x[0] = 1']):
    results['metrics'][metric].plot.bar(ax=ax, rot=20)
    ax.set(title=title, xlabel='', ylabel='MSE')
plt.show()


## Optional: Verstärkung vs. Gedächtniseffekt

Mit `RUN_MATCHED_META_LR = True` wird das Experiment mit frischen Modellen und genau demselben Stream wiederholt. Nur für Momentum gilt dann `momentum_meta_lr = META_LR * (1 - μ)`. Damit ist die Verstärkung für einen konstanten Meta-Gradienten langfristig angeglichen; transient sind beide Regeln weiterhin verschieden. SGD und normales IDBD bleiben unverändert.

Für eine belastbare Aussage mehrere Seeds und Meta-Lernraten testen. Eine Alternative zum Hauptvergleich ist normales IDBD mit `META_LR / (1 - μ)` gegen Momentum mit `META_LR`. Kleinere Lernraten zählen nur zusammen mit kleineren Distraktor-Gewichten und besserem sauberem Testfehler als Verbesserung.

In [ ]:
matched_results = None
if RUN_MATCHED_META_LR:
    matched_results = compare(**config, momentum_meta_lr=META_LR * (1 - BETA_MOMENTUM))
    display(pd.concat({'same meta_lr': results['metrics'],
                       'matched long-run gain': matched_results['metrics']},
                      names=['comparison', 'model']))
    plot_overview(matched_results, 'Matched long-run meta-update gain')
else:
    print('Optionaler Vergleich deaktiviert: RUN_MATCHED_META_LR = True setzen und erneut ausführen.')
